# Fraud detection inference simulation

This notebook demonstrates how to load the autoencoder weights and score a new transaction. It mirrors the architecture in `anomaly_detection_with_autoencoders.ipynb`.

**Prerequisite:** run the training notebook first and make sure `ae_anomaly.weights.h5` is available in this notebook's working directory.

The input must use the training feature order shown below and the same preprocessing: the three numeric features are standardized, while the four binary features are left as 0/1. The training notebook does not save the fitted scaler or a production-calibrated threshold, so this demo reports a reconstruction score only unless you provide a separately calibrated threshold.

In [6]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

import numpy as np
from keras import Model
from keras.layers import Dense, Input

WEIGHTS_PATH = Path("artifacts/ae_anomaly.weights.h5")
if not WEIGHTS_PATH.is_file():
    raise FileNotFoundError(
        f"Weights not found: {WEIGHTS_PATH.resolve()}. "
        "Run the training notebook first and place the generated weights file "
        "in this notebook's working directory."
    )

## Recreate the architecture of trained model and load its weights
Keep all architecture identical here, so the saved weights are interpreted by the same architecture.

In [7]:
CODE_DIM = 2
INPUT_SHAPE = 7

input_layer = Input(shape=(INPUT_SHAPE,))
x = Dense(64, activation="relu")(input_layer)
x = Dense(16, activation="relu")(x)
code = Dense(CODE_DIM, activation="relu")(x)
x = Dense(16, activation="relu")(code)
x = Dense(64, activation="relu")(x)
output_layer = Dense(INPUT_SHAPE, activation="relu")(x)

ae_production = Model(input_layer, output_layer, name="frauddetections")
ae_production.load_weights(WEIGHTS_PATH)
print("Model weights loaded successfully. Ready for inference.")

Model weights loaded successfully. Ready for inference.


## Score a new transaction

Feature order: `distance_from_home`, `distance_from_last_transaction`, `ratio_to_median_purchase_price`, `repeat_retailer`, `used_chip`, `used_pin_number`, `online_order`. The numeric values below are illustrative standardized values; binary values remain 0 or 1.

In [8]:
FEATURE_NAMES = [
    "distance_from_home",
    "distance_from_last_transaction",
    "ratio_to_median_purchase_price",
    "repeat_retailer",
    "used_chip",
    "used_pin_number",
    "online_order",
]

new_transaction_scaled = np.array(
    [[0.5, -1.2, 3.4, 1.0, 0.0, 0.0, 1.0]],
    dtype=np.float32,
)
if new_transaction_scaled.shape[1] != INPUT_SHAPE:
    raise ValueError(f"Expected {INPUT_SHAPE} features, got {new_transaction_scaled.shape[1]}.")

reconstruction = ae_production.predict(new_transaction_scaled, verbose=0)
reconstruction_error = np.mean(
    np.abs(reconstruction - new_transaction_scaled),
    axis=1,
)
print(f"Reconstruction error: {reconstruction_error[0]:.4f}")

Reconstruction error: 0.6580


## Optional threshold-based flag

Set `THRESHOLD` only after calibrating it with an appropriate validation dataset. The training notebook explores thresholds on its test set, so that value should not be treated as an unbiased production threshold.

In [10]:
THRESHOLD = 0.150329  # here threshold with maximum recall from test set

if THRESHOLD is None:
    print("No threshold configured; reporting the anomaly score only.")
else:
    is_anomaly = reconstruction_error[0] > THRESHOLD
    print("anomaly" if is_anomaly else "genuine")

anomaly
